# 03 — Security Control ROI & Sensitivity Analysis

## What we're doing and why

The previous notebooks established *what the risk is* in financial terms. This notebook answers the next question: **what is a security investment actually worth?**

The framing matters. Security spending is often justified by compliance requirements or incident fear. Neither is a capital allocation argument. The right question is:

> *If we invest $X in this control, what is the expected reduction in annual financial exposure, and what is the return on that investment?*

### How we model control effectiveness

Controls reduce the **Threat Event Frequency** — they make it harder for attackers to succeed, reducing the rate of effective incidents. A 40% effective control means 40% fewer successful events per year.

```
Controlled TEF = Baseline TEF × (1 − control_effectiveness)
ROI = (Risk Reduction − Control Cost) / Control Cost
```

We test effectiveness levels from 0% to 80% against a fixed $1M annual control investment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

np.random.seed(42)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

SIMULATIONS  = 10_000
LOSS_MEAN    = 14.5
LOSS_SIGMA   = 0.7
BASE_TEF     = (1, 3, 6)      # Ransomware baseline
CONTROL_COST = 1_000_000      # $1M/yr control investment

CONTROL_LEVELS = [0.0, 0.20, 0.40, 0.60, 0.80]

In [ ]:
# ── Risk engine ───────────────────────────────────────────────────────────────

def simulate_ale(tef_range, loss_mean, loss_sigma, n=10_000):
    tef  = np.random.triangular(*tef_range, n)
    loss = np.random.lognormal(loss_mean, loss_sigma, n)
    return tef * loss


def apply_control(tef_range, reduction):
    return tuple(x * (1 - reduction) for x in tef_range)


def fmt(v):
    return f"${v/1e6:.2f}M"

In [ ]:
# ── Baseline ──────────────────────────────────────────────────────────────────

baseline        = simulate_ale(BASE_TEF, LOSS_MEAN, LOSS_SIGMA, SIMULATIONS)
baseline_median = np.percentile(baseline, 50)
baseline_p90    = np.percentile(baseline, 90)

print(f"Baseline Median Loss: {fmt(baseline_median)}")
print(f"Baseline VaR-90:      {fmt(baseline_p90)}")

In [ ]:
# ── Control sensitivity sweep ─────────────────────────────────────────────────

rows = []
for lvl in CONTROL_LEVELS:
    c_ale    = simulate_ale(apply_control(BASE_TEF, lvl), LOSS_MEAN, LOSS_SIGMA, SIMULATIONS)
    c_median = np.percentile(c_ale, 50)
    c_p90    = np.percentile(c_ale, 90)
    red      = baseline_median - c_median
    roi      = (red - CONTROL_COST) / CONTROL_COST
    rows.append({
        "Effectiveness":          f"{lvl:.0%}",
        "Controlled Median ($M)": round(c_median / 1e6, 2),
        "Risk Reduction ($M)":    round(red / 1e6, 2),
        "ROI":                    round(roi, 2),
        "Justified":              "Yes" if roi > 0 else "No",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── ROI curve ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

eff_vals = [float(r[:-1]) / 100 for r in df["Effectiveness"]]
roi_vals = df["ROI"].tolist()
red_vals = df["Risk Reduction ($M)"].tolist()

# Left: ROI curve
axes[0].plot(eff_vals, roi_vals, marker="o", color="#A100FF", linewidth=2, markersize=7)
axes[0].axhline(0, color="#FF4B4B", linestyle="--", linewidth=1, label="Break-even (ROI = 0)")
axes[0].fill_between(eff_vals, roi_vals, 0,
                     where=[r > 0 for r in roi_vals], alpha=0.12, color="#00C4B4", label="Positive ROI")
axes[0].fill_between(eff_vals, roi_vals, 0,
                     where=[r <= 0 for r in roi_vals], alpha=0.12, color="#FF4B4B", label="Negative ROI")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].set_xlabel("Control Effectiveness", fontsize=11)
axes[0].set_ylabel("ROI (x)", fontsize=11)
axes[0].set_title("ROI vs Control Effectiveness", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)

# Right: Risk reduction bar chart
colors = ["#FF4B4B" if r < 1.0 else "#00C4B4" for r in red_vals]
axes[1].bar([f"{e:.0%}" for e in eff_vals], red_vals, color=colors, alpha=0.85)
axes[1].axhline(CONTROL_COST / 1e6, color="#FFA500", linestyle="--", linewidth=1.5,
                label=f"Control cost ${CONTROL_COST/1e6:.1f}M")
axes[1].set_xlabel("Control Effectiveness", fontsize=11)
axes[1].set_ylabel("Risk Reduction ($M/yr)", fontsize=11)
axes[1].set_title("Annual Risk Reduction by Effectiveness", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
# ── Master insight ────────────────────────────────────────────────────────────

best = df.loc[df["ROI"].idxmax()]
breakeven = df[df["Justified"] == "Yes"].iloc[0]

print("=== CONTROL ROI SUMMARY ===")
print(f"Break-even effectiveness:  {breakeven['Effectiveness']}")
print(f"Best ROI at:               {best['Effectiveness']} effectiveness")
print(f"Max risk reduction:        ${best['Risk Reduction ($M)']:.2f}M/yr")
print(f"Max ROI:                   {best['ROI']:.2f}x")

## Key Takeaways

| Insight | Implication |
|---------|-------------|
| ROI breaks even at ~20% control effectiveness | Even modest controls can be financially justified |
| At 80% effectiveness, ROI reaches ~4x | High-quality controls are exceptional financial investments |
| The relationship is linear in risk reduction, but the ROI curve is steep | Small improvements in effectiveness have outsized financial return |

**The executive takeaway:** Security investment decisions should not be binary (buy / don't buy). They should be sensitivity-based: *at what level of effectiveness does this control pay for itself, and what is our confidence in achieving that level?*

This reframes the vendor conversation: instead of asking *"does this tool work?"*, ask *"what effectiveness level does this tool reliably deliver, and what does that imply for our ROI?"*

---

**Series complete.** For a client-facing interactive version of this analysis with FAIR engine, 5-scenario modeling, and PDF reporting, see the [CRQ Dashboard](../../CRQ).